In [1]:

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)



from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import LSTM


from tensorflow.keras.layers import (
    Conv1D,
    Dense,
    Dropout,
    GlobalMaxPooling1D
)

from tensorflow.keras.optimizers import Adam
from scipy.stats import t


In [2]:

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

DATASET_PATH = "url_features_extracted1(100K).csv"

BATCH_SIZE = 16
LEARNING_RATE = 0.001
EPOCHS = 30
SEEDS = [42, 3, 7, 72, 82]

# ------------------------------------------------------------
# REPRODUCIBILITY
# ------------------------------------------------------------

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(42)

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

data = pd.read_csv(DATASET_PATH)

print("Dataset Shape:", data.shape)
print(data.head())

print("\nColumns:")
print(data.columns.tolist())

# ------------------------------------------------------------
# VERIFY TARGET COLUMN
# ------------------------------------------------------------

TARGET = "ClassLabel"

if TARGET not in data.columns:
    raise ValueError(f"{TARGET} not found in dataset.")

# ------------------------------------------------------------
# HANDLE MISSING VALUES
# ------------------------------------------------------------

numeric_cols = data.select_dtypes(include=np.number).columns.tolist()
categorical_cols = data.select_dtypes(exclude=np.number).columns.tolist()

# Remove target from feature lists
if TARGET in numeric_cols:
    numeric_cols.remove(TARGET)

if TARGET in categorical_cols:
    categorical_cols.remove(TARGET)

# Fill missing values
data[numeric_cols] = data[numeric_cols].fillna(
    data[numeric_cols].mean()
)

data[categorical_cols] = data[categorical_cols].fillna("unknown")

# ------------------------------------------------------------
# ENCODE CATEGORICAL FEATURES
# ------------------------------------------------------------

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))

# ------------------------------------------------------------
# ENCODE TARGET IF NEEDED
# ------------------------------------------------------------

if data[TARGET].dtype == object:
    target_encoder = LabelEncoder()
    data[TARGET] = target_encoder.fit_transform(data[TARGET])

# ------------------------------------------------------------
# CHECK CLASS DISTRIBUTION
# ------------------------------------------------------------

print("\nClass Distribution:")
print(data[TARGET].value_counts())

# Remove classes having fewer than 2 samples
class_counts = data[TARGET].value_counts()

valid_classes = class_counts[class_counts >= 2].index

data = data[data[TARGET].isin(valid_classes)].reset_index(drop=True)

print("\nAfter Removing Rare Classes:")
print(data[TARGET].value_counts())

# ------------------------------------------------------------
# FEATURES & LABELS
# ------------------------------------------------------------

X = data.drop(columns=[TARGET])
y = data[TARGET]

# ------------------------------------------------------------
# TRAIN TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ------------------------------------------------------------
# NORMALIZATION
# ------------------------------------------------------------

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ------------------------------------------------------------
# RESHAPE FOR CNN
# ------------------------------------------------------------

X_train = X_train.reshape(
    X_train.shape[0],
    X_train.shape[1],
    1
)

X_test = X_test.reshape(
    X_test.shape[0],
    X_test.shape[1],
    1
)

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("\nTraining Samples :", len(X_train))
print("Testing Samples  :", len(X_test))

print("\nTraining Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

print("\nDone.")

Dataset Shape: (101219, 18)
                                               URL  url_length  \
0  https://keraekken-loagginnusa.godaddysites.com/          47   
1         https://metamsk01lgiix.godaddysites.com/          40   
2                          http://myglobaltech.in/          23   
3                   http://djtool-for-spotify.com/          30   
4  https://scearmcoommunnlty.com/invent/freind/get          47   

   has_ip_address  dot_count  https_flag  url_entropy  token_count  \
0               0          2           1     4.250669            6   
1               0          2           1     4.196439            6   
2               0          1           0     3.936180            5   
3               0          1           0     3.894740            5   
4               0          1           1     4.143127            7   

   subdomain_count  query_param_count  tld_length  path_length  \
0                1                  1           3            1   
1                1    

# CNN

In [6]:



def create_model(input_shape):

    model = Sequential()

    model.add(
        Input(shape=input_shape)
    )

    model.add(
        Conv1D(
            filters=64,
            kernel_size=3,
            activation="relu"
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Conv1D(
            filters=32,
            kernel_size=3,
            activation="relu"
        )
    )

    model.add(
        GlobalMaxPooling1D()
    )

    model.add(
        Dense(
            32,
            activation="relu"
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Dense(
            1,
            activation="sigmoid"
        )
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc")
        ]
    )

    return model

In [7]:
# ============================================================
# PART 2
# CENTRALIZED TRAINING (1D-CNN)
# ============================================================

results = []

for seed in SEEDS:

    print("=" * 70)
    print(f"Running Seed : {seed}")
    print("=" * 70)

    # Reproducibility
    set_seed(seed)

    # Build 1D-CNN Model
    model = create_model(
        (X_train.shape[1], X_train.shape[2])
    )

    # Train Model
    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        verbose=1
    )

    # =====================================================
    # Test Loss
    # =====================================================

    evaluation = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )

    test_loss = evaluation[0]

    # =====================================================
    # Prediction
    # =====================================================

    y_prob = model.predict(
        X_test,
        verbose=0
    ).flatten()

    y_pred = (y_prob >= 0.5).astype(int)


    # =====================================================
    # Performance Metrics
    # =====================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()

    specificity = tn / (tn + fp)


    # =====================================================
    # Save Results
    # =====================================================

    results.append({

        "Seed": seed,

        "Loss": test_loss,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "MCC": mcc,

        "AUC": auc,

        "Specificity": specificity

    })


    # =====================================================
    # Print Results
    # =====================================================

    print("\nTest Performance")
    print("-" * 40)

    print(f"Loss        : {test_loss:.4f}")
    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Precision   : {precision:.4f}")
    print(f"Recall      : {recall:.4f}")
    print(f"F1-score    : {f1:.4f}")
    print(f"MCC         : {mcc:.4f}")
    print(f"AUC         : {auc:.4f}")
    print(f"Specificity : {specificity:.4f}")

    print()

Running Seed : 42


I0000 00:00:1785938982.380917      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/30
  63/5061 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - accuracy: 0.6384 - auc: 0.6447 - loss: 0.6473 

I0000 00:00:1785938989.741819     108 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


5061/5061 ━━━━━━━━━━━━━━━━━━━━ 24s 4ms/step - accuracy: 0.9810 - auc: 0.9955 - loss: 0.0695 - val_accuracy: 0.9946 - val_auc: 0.9997 - val_loss: 0.0123
Epoch 2/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.9932 - auc: 0.9991 - loss: 0.0231 - val_accuracy: 0.9953 - val_auc: 0.9998 - val_loss: 0.0090
Epoch 3/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.9937 - auc: 0.9993 - loss: 0.0181 - val_accuracy: 0.9953 - val_auc: 0.9998 - val_loss: 0.0081
Epoch 4/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.9945 - auc: 0.9994 - loss: 0.0163 - val_accuracy: 0.9959 - val_auc: 0.9998 - val_loss: 0.0078
Epoch 5/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.9949 - auc: 0.9994 - loss: 0.0156 - val_accuracy: 0.9959 - val_auc: 0.9999 - val_loss: 0.0074
Epoch 6/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.9953 - auc: 0.9995 - loss: 0.0133 - val_accuracy: 0.9970 - val_auc: 0.9998 - val_loss: 0.0060
Epoch 7/30
5061/5061 ━━━━━━━━━━━━

In [8]:
# ============================================================
# MEAN ± SD + 95% CONFIDENCE INTERVAL
# ============================================================

results_df = pd.DataFrame(results)

print("\nResults from 5 Seeds")
print(results_df)


# ------------------------------------------------------------
# Function to calculate Mean, SD and 95% CI
# ------------------------------------------------------------

def calculate_statistics(values):

    values = np.array(values)

    n = len(values)

    mean = np.mean(values)

    sd = np.std(values, ddof=1)

    se = sd / np.sqrt(n)

    t_value = t.ppf(0.975, df=n-1)

    margin = t_value * se

    lower = mean - margin

    upper = mean + margin

    return mean, sd, lower, upper


# ------------------------------------------------------------
# Compute statistics for each metric
# ------------------------------------------------------------

summary = []

metrics = [
    "Loss",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "AUC",
    "Specificity"
]


for metric in metrics:

    mean, sd, lower, upper = calculate_statistics(
        results_df[metric]
    )

    summary.append({

        "Metric": metric,

        "Mean": mean,

        "SD": sd,

        "Mean ± SD": f"{mean:.4f} ± {sd:.4f}",

        "95% CI": f"[{lower:.4f}, {upper:.4f}]"

    })


summary_df = pd.DataFrame(summary)


# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\n")
print("="*70)
print("Performance Summary (5 Seeds)")
print("="*70)

print(summary_df)


# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

results_df.to_csv(
    "Centralized_5Seed_Results.csv",
    index=False
)

summary_df.to_csv(
    "Centralized_Performance_Summary.csv",
    index=False
)


print("\nCSV files saved successfully.")


Results from 5 Seeds
   Seed      Loss  Accuracy  Precision    Recall        F1       MCC  \
0    42  0.004627  0.998320   0.998400  0.997070  0.997734  0.996401   
1     3  0.005862  0.997728   0.994434  0.999467  0.996944  0.995144   
2     7  0.005190  0.998567   0.997472  0.998668  0.998070  0.996931   
3    72  0.006383  0.997876   0.995092  0.999201  0.997142  0.995457   
4    82  0.005573  0.997629   0.994826  0.998801  0.996810  0.994928   

        AUC  Specificity  
0  0.999986     0.999058  
1  0.999961     0.996702  
2  0.999967     0.998508  
3  0.999962     0.997095  
4  0.999961     0.996938  


Performance Summary (5 Seeds)
        Metric      Mean        SD        Mean ± SD            95% CI
0         Loss  0.005527  0.000665  0.0055 ± 0.0007  [0.0047, 0.0064]
1     Accuracy  0.998024  0.000403  0.9980 ± 0.0004  [0.9975, 0.9985]
2    Precision  0.996045  0.001773  0.9960 ± 0.0018  [0.9938, 0.9982]
3       Recall  0.998641  0.000934  0.9986 ± 0.0009  [0.9975, 0.9998]
4

# LSTM

In [3]:



def create_model():

    model = Sequential()

    model.add(
        Input(
            shape=(
                X_train.shape[1],
                X_train.shape[2]
            )
        )
    )

    model.add(
        LSTM(
            128,
            return_sequences=True
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        LSTM(
            64
        )
    )

    model.add(
        Dropout(0.5
        )
    )

    model.add(
        Dense(
            1,
            activation='sigmoid'
        )
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),

        loss='binary_crossentropy',

        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc')
        ]
    )

    return model

In [4]:
# ============================================================
# PART 2
# CENTRALIZED TRAINING (LSTM)
# ============================================================

results = []

for seed in SEEDS:

    print("=" * 70)
    print(f"Running Seed : {seed}")
    print("=" * 70)

    # Reproducibility
    set_seed(seed)

    # =====================================================
    # Build LSTM Model
    # =====================================================

    model = create_model()


    # =====================================================
    # Train Model
    # =====================================================

    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        verbose=1
    )


    # =====================================================
    # Test Loss
    # =====================================================

    test_loss, test_accuracy, test_auc = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )


    # =====================================================
    # Prediction
    # =====================================================

    y_prob = model.predict(
        X_test,
        verbose=0
    ).flatten()

    y_pred = (y_prob >= 0.5).astype(int)


    # =====================================================
    # Performance Metrics
    # =====================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )


    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()


    specificity = tn / (tn + fp)


    # =====================================================
    # Save Results
    # =====================================================

    results.append({

        "Seed": seed,

        "Loss": test_loss,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "MCC": mcc,

        "AUC": auc,

        "Specificity": specificity

    })


    # =====================================================
    # Print Results
    # =====================================================

    print("\nTest Performance")
    print("-" * 40)

    print(f"Loss        : {test_loss:.4f}")
    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Precision   : {precision:.4f}")
    print(f"Recall      : {recall:.4f}")
    print(f"F1-score    : {f1:.4f}")
    print(f"MCC         : {mcc:.4f}")
    print(f"AUC         : {auc:.4f}")
    print(f"Specificity : {specificity:.4f}")

    print()

Running Seed : 42


I0000 00:00:1785995532.363444      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 43s 7ms/step - accuracy: 0.9825 - auc: 0.9971 - loss: 0.0556 - val_accuracy: 0.9961 - val_auc: 0.9994 - val_loss: 0.0124
Epoch 2/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 37s 7ms/step - accuracy: 0.9958 - auc: 0.9995 - loss: 0.0131 - val_accuracy: 0.9983 - val_auc: 0.9999 - val_loss: 0.0056
Epoch 3/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 37s 7ms/step - accuracy: 0.9977 - auc: 0.9997 - loss: 0.0079 - val_accuracy: 0.9982 - val_auc: 0.9999 - val_loss: 0.0053
Epoch 4/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 38s 7ms/step - accuracy: 0.9983 - auc: 0.9997 - loss: 0.0063 - val_accuracy: 0.9981 - val_auc: 0.9998 - val_loss: 0.0054
Epoch 5/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 37s 7ms/step - accuracy: 0.9981 - auc: 0.9998 - loss: 0.0059 - val_accuracy: 0.9987 - val_auc: 0.9998 - val_loss: 0.0039
Epoch 6/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 37s 7ms/step - accuracy: 0.9984 - auc: 0.9998 - loss: 0.0052 - val_accuracy: 0.9982 - val_auc: 0.9998 - val_loss: 0.0050
Epoch 7/30
5061/5061 ━

In [5]:
# ============================================================
# MEAN ± SD + 95% CONFIDENCE INTERVAL
# ============================================================

results_df = pd.DataFrame(results)

print("\nResults from 5 Seeds")
print(results_df)


# ------------------------------------------------------------
# Function to calculate Mean, SD and 95% CI
# ------------------------------------------------------------

def calculate_statistics(values):

    values = np.array(values)

    n = len(values)

    mean = np.mean(values)

    sd = np.std(values, ddof=1)

    se = sd / np.sqrt(n)

    t_value = t.ppf(0.975, df=n-1)

    margin = t_value * se

    lower = mean - margin

    upper = mean + margin

    return mean, sd, lower, upper


# ------------------------------------------------------------
# Compute statistics for each metric
# ------------------------------------------------------------

summary = []

metrics = [
    "Loss",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "AUC",
    "Specificity"
]


for metric in metrics:

    mean, sd, lower, upper = calculate_statistics(
        results_df[metric]
    )

    summary.append({

        "Metric": metric,

        "Mean": mean,

        "SD": sd,

        "Mean ± SD": f"{mean:.4f} ± {sd:.4f}",

        "95% CI": f"[{lower:.4f}, {upper:.4f}]"

    })


summary_df = pd.DataFrame(summary)


# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\n")
print("="*70)
print("Performance Summary (5 Seeds)")
print("="*70)

print(summary_df)


# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

results_df.to_csv(
    "Centralized_5Seed_Results.csv",
    index=False
)

summary_df.to_csv(
    "Centralized_Performance_Summary.csv",
    index=False
)


print("\nCSV files saved successfully.")


Results from 5 Seeds
   Seed      Loss  Accuracy  Precision    Recall        F1       MCC  \
0    42  0.007345  0.999111   0.998934  0.998668  0.998801  0.998095   
1     3  0.004299  0.999210   0.998536  0.999334  0.998935  0.998307   
2     7  0.003029  0.999210   0.998403  0.999467  0.998935  0.998307   
3    72  0.003155  0.999308   0.998536  0.999600  0.999068  0.998519   
4    82  0.006155  0.998666   0.996812  0.999600  0.998204  0.997146   

        AUC  Specificity  
0  0.999983     0.999372  
1  0.999993     0.999136  
2  0.999995     0.999058  
3  0.999996     0.999136  
4  0.999994     0.998116  


Performance Summary (5 Seeds)
        Metric      Mean        SD        Mean ± SD            95% CI
0         Loss  0.004797  0.001898  0.0048 ± 0.0019  [0.0024, 0.0072]
1     Accuracy  0.999101  0.000253  0.9991 ± 0.0003  [0.9988, 0.9994]
2    Precision  0.998244  0.000825  0.9982 ± 0.0008  [0.9972, 0.9993]
3       Recall  0.999334  0.000388  0.9993 ± 0.0004  [0.9989, 0.9998]
4

# BiLSTM

In [3]:

from tensorflow.keras.layers import Bidirectional



def create_model():

    model = Sequential()

    model.add(
        Input(
            shape=(
                X_train.shape[1],
                X_train.shape[2]
            )
        )
    )

    model.add(
        Bidirectional(
            LSTM(
                128,
                return_sequences=True
            )
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Bidirectional(
            LSTM(
                64
            )
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Dense(
            1,
            activation='sigmoid'
        )
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),

        loss='binary_crossentropy',

        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc')
        ]
    )

    return model

In [4]:
# ============================================================
# PART 2
# CENTRALIZED BiTRAINING (BiLSTM)
# ============================================================

results = []

for seed in SEEDS:

    print("=" * 70)
    print(f"Running Seed : {seed}")
    print("=" * 70)

    # Reproducibility
    set_seed(seed)

    # =====================================================
    # Build BiLSTM Model
    # =====================================================

    model = create_model()


    # =====================================================
    # Train Model
    # =====================================================

    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        verbose=1
    )


    # =====================================================
    # Test Loss
    # =====================================================

    test_loss, test_accuracy, test_auc = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )


    # =====================================================
    # Prediction
    # =====================================================

    y_prob = model.predict(
        X_test,
        verbose=0
    ).flatten()

    y_pred = (y_prob >= 0.5).astype(int)


    # =====================================================
    # Performance Metrics
    # =====================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )


    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()


    specificity = tn / (tn + fp)


    # =====================================================
    # Save Results
    # =====================================================

    results.append({

        "Seed": seed,

        "Loss": test_loss,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "MCC": mcc,

        "AUC": auc,

        "Specificity": specificity

    })


    # =====================================================
    # Print Results
    # =====================================================

    print("\nTest Performance")
    print("-" * 40)

    print(f"Loss        : {test_loss:.4f}")
    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Precision   : {precision:.4f}")
    print(f"Recall      : {recall:.4f}")
    print(f"F1-score    : {f1:.4f}")
    print(f"MCC         : {mcc:.4f}")
    print(f"AUC         : {auc:.4f}")
    print(f"Specificity : {specificity:.4f}")

    print()

Running Seed : 42
Epoch 1/30
5061/5061 [==============================] - 87s 17ms/step - loss: 0.0460 - accuracy: 0.9843 - auc: 0.9981 - val_loss: 0.0088 - val_accuracy: 0.9973 - val_auc: 0.9998
Epoch 2/30
5061/5061 [==============================] - 78s 15ms/step - loss: 0.0105 - accuracy: 0.9966 - auc: 0.9997 - val_loss: 0.0099 - val_accuracy: 0.9969 - val_auc: 0.9999
Epoch 3/30
5061/5061 [==============================] - 79s 16ms/step - loss: 0.0074 - accuracy: 0.9978 - auc: 0.9997 - val_loss: 0.0070 - val_accuracy: 0.9967 - val_auc: 0.9998
Epoch 4/30
5061/5061 [==============================] - 79s 16ms/step - loss: 0.0055 - accuracy: 0.9981 - auc: 0.9998 - val_loss: 0.0091 - val_accuracy: 0.9972 - val_auc: 0.9994
Epoch 5/30
5061/5061 [==============================] - 79s 16ms/step - loss: 0.0058 - accuracy: 0.9983 - auc: 0.9997 - val_loss: 0.0042 - val_accuracy: 0.9984 - val_auc: 0.9999
Epoch 6/30
5061/5061 [==============================] - 79s 16ms/step - loss: 0.0052 - accur

In [5]:
# ============================================================
# MEAN ± SD + 95% CONFIDENCE INTERVAL
# ============================================================

results_df = pd.DataFrame(results)

print("\nResults from 5 Seeds")
print(results_df)


# ------------------------------------------------------------
# Function to calculate Mean, SD and 95% CI
# ------------------------------------------------------------

def calculate_statistics(values):

    values = np.array(values)

    n = len(values)

    mean = np.mean(values)

    sd = np.std(values, ddof=1)

    se = sd / np.sqrt(n)

    t_value = t.ppf(0.975, df=n-1)

    margin = t_value * se

    lower = mean - margin

    upper = mean + margin

    return mean, sd, lower, upper


# ------------------------------------------------------------
# Compute statistics for each metric
# ------------------------------------------------------------

summary = []

metrics = [
    "Loss",      
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "AUC",
    "Specificity"
]


for metric in metrics:

    mean, sd, lower, upper = calculate_statistics(
        results_df[metric]
    )

    summary.append({

        "Metric": metric,

        "Mean": mean,

        "SD": sd,

        "Mean ± SD": f"{mean:.4f} ± {sd:.4f}",

        "95% CI": f"[{lower:.4f}, {upper:.4f}]"

    })


summary_df = pd.DataFrame(summary)


# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\n")
print("="*70)
print("Performance Summary (5 Seeds)")
print("="*70)

print(summary_df)


# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

results_df.to_csv(
    "Centralized_5Seed_Results.csv",
    index=False
)

summary_df.to_csv(
    "Centralized_Performance_Summary.csv",
    index=False
)


print("\nCSV files saved successfully.")


Results from 5 Seeds
   Seed      Loss  Accuracy  Precision    Recall        F1       MCC  \
0    42  0.002612  0.999061   0.998005  0.999467  0.998736  0.997990   
1     3  0.002255  0.999358   0.998802  0.999467  0.999135  0.998624   
2     7  0.003412  0.999210   0.998403  0.999467  0.998935  0.998307   
3    72  0.003824  0.999160   0.998801  0.998934  0.998868  0.998201   
4    82  0.002476  0.999308   0.998536  0.999600  0.999068  0.998519   

        AUC  Specificity  
0  0.999995     0.998822  
1  0.999998     0.999293  
2  0.999997     0.999058  
3  0.999992     0.999293  
4  0.999996     0.999136  


Performance Summary (5 Seeds)
        Metric      Mean        SD        Mean ± SD            95% CI
0         Loss  0.002916  0.000669  0.0029 ± 0.0007  [0.0021, 0.0037]
1     Accuracy  0.999220  0.000118  0.9992 ± 0.0001  [0.9991, 0.9994]
2    Precision  0.998510  0.000331  0.9985 ± 0.0003  [0.9981, 0.9989]
3       Recall  0.999387  0.000260  0.9994 ± 0.0003  [0.9991, 0.9997]
4

# GRU

In [9]:

from tensorflow.keras.layers import GRU



def create_model():

    model = Sequential()

    model.add(
        Input(
            shape=(
                X_train.shape[1],
                X_train.shape[2]
            )
        )
    )

    model.add(
        GRU(
            128,
            return_sequences=True
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        GRU(
            64
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Dense(
            1,
            activation='sigmoid'
        )
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),

        loss='binary_crossentropy',

        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc')
        ]
    )

    return model

In [10]:
# ============================================================
# PART 2
# CENTRALIZED BiTRAINING (GRU)
# ============================================================

results = []

for seed in SEEDS:

    print("=" * 70)
    print(f"Running Seed : {seed}")
    print("=" * 70)

    # Reproducibility
    set_seed(seed)

    # =====================================================
    # Build BiLSTM Model
    # =====================================================

    model = create_model()


    # =====================================================
    # Train Model
    # =====================================================

    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        verbose=1
    )


    # =====================================================
    # Test Loss
    # =====================================================

    test_loss, test_accuracy, test_auc = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )


    # =====================================================
    # Prediction
    # =====================================================

    y_prob = model.predict(
        X_test,
        verbose=0
    ).flatten()

    y_pred = (y_prob >= 0.5).astype(int)


    # =====================================================
    # Performance Metrics
    # =====================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )


    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()


    specificity = tn / (tn + fp)


    # =====================================================
    # Save Results
    # =====================================================

    results.append({

        "Seed": seed,

        "Loss": test_loss,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "MCC": mcc,

        "AUC": auc,

        "Specificity": specificity

    })


    # =====================================================
    # Print Results
    # =====================================================

    print("\nTest Performance")
    print("-" * 40)

    print(f"Loss        : {test_loss:.4f}")
    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Precision   : {precision:.4f}")
    print(f"Recall      : {recall:.4f}")
    print(f"F1-score    : {f1:.4f}")
    print(f"MCC         : {mcc:.4f}")
    print(f"AUC         : {auc:.4f}")
    print(f"Specificity : {specificity:.4f}")

    print()

Running Seed : 42
Epoch 1/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 39s 7ms/step - accuracy: 0.9902 - auc: 0.9987 - loss: 0.0319 - val_accuracy: 0.9973 - val_auc: 0.9997 - val_loss: 0.0086
Epoch 2/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 37s 7ms/step - accuracy: 0.9967 - auc: 0.9996 - loss: 0.0108 - val_accuracy: 0.9976 - val_auc: 0.9999 - val_loss: 0.0084
Epoch 3/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 37s 7ms/step - accuracy: 0.9976 - auc: 0.9997 - loss: 0.0078 - val_accuracy: 0.9987 - val_auc: 0.9999 - val_loss: 0.0044
Epoch 4/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 36s 7ms/step - accuracy: 0.9981 - auc: 0.9997 - loss: 0.0065 - val_accuracy: 0.9990 - val_auc: 0.9998 - val_loss: 0.0035
Epoch 5/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 37s 7ms/step - accuracy: 0.9982 - auc: 0.9998 - loss: 0.0053 - val_accuracy: 0.9988 - val_auc: 0.9998 - val_loss: 0.0045
Epoch 6/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 36s 7ms/step - accuracy: 0.9984 - auc: 0.9999 - loss: 0.0047 - val_accuracy: 0.9990 - val_auc: 0.9998 - val_loss: 0.0036
Epoc

In [11]:
# ============================================================
# MEAN ± SD + 95% CONFIDENCE INTERVAL
# ============================================================

results_df = pd.DataFrame(results)

print("\nResults from 5 Seeds")
print(results_df)


# ------------------------------------------------------------
# Function to calculate Mean, SD and 95% CI
# ------------------------------------------------------------

def calculate_statistics(values):

    values = np.array(values)

    n = len(values)

    mean = np.mean(values)

    sd = np.std(values, ddof=1)

    se = sd / np.sqrt(n)

    t_value = t.ppf(0.975, df=n-1)

    margin = t_value * se

    lower = mean - margin

    upper = mean + margin

    return mean, sd, lower, upper


# ------------------------------------------------------------
# Compute statistics for each metric
# ------------------------------------------------------------

summary = []

metrics = [
    "Loss",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "AUC",
    "Specificity"
]


for metric in metrics:

    mean, sd, lower, upper = calculate_statistics(
        results_df[metric]
    )

    summary.append({

        "Metric": metric,

        "Mean": mean,

        "SD": sd,

        "Mean ± SD": f"{mean:.4f} ± {sd:.4f}",

        "95% CI": f"[{lower:.4f}, {upper:.4f}]"

    })


summary_df = pd.DataFrame(summary)


# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\n")
print("="*70)
print("Performance Summary (5 Seeds)")
print("="*70)

print(summary_df)


# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

results_df.to_csv(
    "Centralized_5Seed_Results.csv",
    index=False
)

summary_df.to_csv(
    "Centralized_Performance_Summary.csv",
    index=False
)


print("\nCSV files saved successfully.")


Results from 5 Seeds
   Seed      Loss  Accuracy  Precision    Recall        F1       MCC  \
0    42  0.004522  0.998765   0.997606  0.999068  0.998336  0.997355   
1     3  0.004107  0.998716   0.998269  0.998269  0.998269  0.997248   
2     7  0.003601  0.999160   0.998271  0.999467  0.998869  0.998201   
3    72  0.003619  0.999259   0.998669  0.999334  0.999001  0.998413   
4    82  0.009417  0.998123   0.997204  0.997736  0.997470  0.995978   

        AUC  Specificity  
0  0.999991     0.998587  
1  0.999992     0.998979  
2  0.999980     0.998979  
3  0.999994     0.999215  
4  0.999968     0.998351  


Performance Summary (5 Seeds)
        Metric      Mean        SD        Mean ± SD            95% CI
0         Loss  0.005053  0.002469  0.0051 ± 0.0025  [0.0020, 0.0081]
1     Accuracy  0.998805  0.000449  0.9988 ± 0.0004  [0.9982, 0.9994]
2    Precision  0.998004  0.000588  0.9980 ± 0.0006  [0.9973, 0.9987]
3       Recall  0.998775  0.000744  0.9988 ± 0.0007  [0.9979, 0.9997]
4

# BiGRU

In [12]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, GRU, Bidirectional, Dropout, Dense
from tensorflow.keras.optimizers import Adam
import tensorflow as tf


def create_model():

    model = Sequential()

    model.add(
        Input(
            shape=(
                X_train.shape[1],
                X_train.shape[2]
            )
        )
    )

    model.add(
        Bidirectional(
            GRU(
                128,
                return_sequences=True
            )
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Bidirectional(
            GRU(
                64
            )
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Dense(
            1,
            activation='sigmoid'
        )
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),

        loss='binary_crossentropy',

        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc')
        ]
    )

    return model

In [13]:
# ============================================================
# PART 2
# CENTRALIZED BiTRAINING (BiGRU)
# ============================================================

results = []

for seed in SEEDS:

    print("=" * 70)
    print(f"Running Seed : {seed}")
    print("=" * 70)

    # Reproducibility
    set_seed(seed)

    # =====================================================
    # Build BiLSTM Model
    # =====================================================

    model = create_model()


    # =====================================================
    # Train Model
    # =====================================================

    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        verbose=1
    )


    # =====================================================
    # Test Loss
    # =====================================================

    test_loss, test_accuracy, test_auc = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )


    # =====================================================
    # Prediction
    # =====================================================

    y_prob = model.predict(
        X_test,
        verbose=0
    ).flatten()

    y_pred = (y_prob >= 0.5).astype(int)


    # =====================================================
    # Performance Metrics
    # =====================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )


    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()


    specificity = tn / (tn + fp)


    # =====================================================
    # Save Results
    # =====================================================

    results.append({

        "Seed": seed,

        "Loss": test_loss,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "MCC": mcc,

        "AUC": auc,

        "Specificity": specificity

    })


    # =====================================================
    # Print Results
    # =====================================================

    print("\nTest Performance")
    print("-" * 40)

    print(f"Loss        : {test_loss:.4f}")
    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Precision   : {precision:.4f}")
    print(f"Recall      : {recall:.4f}")
    print(f"F1-score    : {f1:.4f}")
    print(f"MCC         : {mcc:.4f}")
    print(f"AUC         : {auc:.4f}")
    print(f"Specificity : {specificity:.4f}")

    print()

Running Seed : 42
Epoch 1/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 56s 11ms/step - accuracy: 0.9916 - auc: 0.9992 - loss: 0.0256 - val_accuracy: 0.9968 - val_auc: 0.9999 - val_loss: 0.0073
Epoch 2/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 57s 11ms/step - accuracy: 0.9966 - auc: 0.9997 - loss: 0.0100 - val_accuracy: 0.9978 - val_auc: 0.9999 - val_loss: 0.0062
Epoch 3/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 61s 12ms/step - accuracy: 0.9976 - auc: 0.9997 - loss: 0.0071 - val_accuracy: 0.9981 - val_auc: 0.9998 - val_loss: 0.0059
Epoch 4/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 61s 12ms/step - accuracy: 0.9980 - auc: 0.9998 - loss: 0.0062 - val_accuracy: 0.9991 - val_auc: 0.9999 - val_loss: 0.0032
Epoch 5/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 60s 12ms/step - accuracy: 0.9984 - auc: 0.9998 - loss: 0.0055 - val_accuracy: 0.9987 - val_auc: 0.9999 - val_loss: 0.0040
Epoch 6/30
5061/5061 ━━━━━━━━━━━━━━━━━━━━ 59s 12ms/step - accuracy: 0.9984 - auc: 0.9998 - loss: 0.0052 - val_accuracy: 0.9987 - val_auc: 0.9996 - val_loss: 0.005

In [14]:
# ============================================================
# MEAN ± SD + 95% CONFIDENCE INTERVAL
# ============================================================

results_df = pd.DataFrame(results)

print("\nResults from 5 Seeds")
print(results_df)


# ------------------------------------------------------------
# Function to calculate Mean, SD and 95% CI
# ------------------------------------------------------------

def calculate_statistics(values):

    values = np.array(values)

    n = len(values)

    mean = np.mean(values)

    sd = np.std(values, ddof=1)

    se = sd / np.sqrt(n)

    t_value = t.ppf(0.975, df=n-1)

    margin = t_value * se

    lower = mean - margin

    upper = mean + margin

    return mean, sd, lower, upper


# ------------------------------------------------------------
# Compute statistics for each metric
# ------------------------------------------------------------

summary = []

metrics = [
    "Loss",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "AUC",
    "Specificity"
]


for metric in metrics:

    mean, sd, lower, upper = calculate_statistics(
        results_df[metric]
    )

    summary.append({

        "Metric": metric,

        "Mean": mean,

        "SD": sd,

        "Mean ± SD": f"{mean:.4f} ± {sd:.4f}",

        "95% CI": f"[{lower:.4f}, {upper:.4f}]"

    })


summary_df = pd.DataFrame(summary)


# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\n")
print("="*70)
print("Performance Summary (5 Seeds)")
print("="*70)

print(summary_df)


# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

results_df.to_csv(
    "Centralized_5Seed_Results.csv",
    index=False
)

summary_df.to_csv(
    "Centralized_Performance_Summary.csv",
    index=False
)


print("\nCSV files saved successfully.")


Results from 5 Seeds
   Seed      Loss  Accuracy  Precision    Recall        F1       MCC  \
0    42  0.006059  0.998716   0.998534  0.998002  0.998268  0.997248   
1     3  0.004380  0.998913   0.997739  0.999334  0.998536  0.997673   
2     7  0.004058  0.999012   0.998005  0.999334  0.998669  0.997884   
3    72  0.002751  0.999259   0.998802  0.999201  0.999001  0.998412   
4    82  0.004626  0.999061   0.997873  0.999600  0.998736  0.997990   

        AUC  Specificity  
0  0.999971     0.999136  
1  0.999991     0.998665  
2  0.999978     0.998822  
3  0.999996     0.999293  
4  0.999988     0.998744  


Performance Summary (5 Seeds)
        Metric      Mean        SD        Mean ± SD            95% CI
0         Loss  0.004375  0.001187  0.0044 ± 0.0012  [0.0029, 0.0058]
1     Accuracy  0.998992  0.000199  0.9990 ± 0.0002  [0.9987, 0.9992]
2    Precision  0.998191  0.000456  0.9982 ± 0.0005  [0.9976, 0.9988]
3       Recall  0.999094  0.000628  0.9991 ± 0.0006  [0.9983, 0.9999]
4